In [4]:
from DWG_to_Excel_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\paiva\temp"
output_folder = r"C:\Users\Usuario\Desktop\paiva\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - Eletromecânico.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - Eletromecânico.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - Eletromecânico.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


In [5]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                  X              Y      rotation
count  12860.000000   12860.000000  12860.000000
mean   35060.961232  128079.736308     25.250389
std      593.172574     951.768149     62.562695
min    33667.553332  125996.367175      0.000000
25%    34622.695128  127293.973600      0.000000
50%    35000.922088  128002.283775      0.000000
75%    35483.685992  128959.993661      0.000000
max    36625.891051  129787.710488    270.000000
<class 'pandas.DataFrame'>
RangeIndex: 12860 entries, 0 to 12859
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   type      12860 non-null  str    
 1   layer     12860 non-null  str    
 2   handle    12860 non-null  str    
 3   name      12860 non-null  str    
 4   X         12860 non-null  float64
 5   Y         12860 non-null  float64
 6   rotation  12860 non-null  float64
 7   UFV       12848 non-null  str    
 8   PST       12848 non-null  str    
 9   INV       12831 non-null  s

In [6]:
mask = (ELM_df['name'] == '1x96p_ponta') | (ELM_df['name'] == '1x96p_meio') | (ELM_df['name'] == '1x64p') | (ELM_df['name'] == '1x32p')
tracker_df = ELM_df[mask].copy()

tracker_df.drop(columns=['handle', 'type', 'QBT', 'DSJ', 'PVM', 'GRP'], inplace=True)

print(tracker_df)
print(tracker_df.describe())
print(tracker_df.info())

               layer   name             X              Y  rotation UFV PST INV
0      EMF_em_pst_21  1x64p  34979.686150  127465.060575     180.0      21   9
1      EMF_em_pst_21  1x64p  34979.686150  127507.562575       0.0      21   9
2      EMF_em_pst_21  1x64p  34985.986150  127465.060575     180.0      21   9
3      EMF_em_pst_21  1x64p  34985.986150  127507.562575       0.0      21   9
4      EMF_em_pst_21  1x64p  34992.286150  127465.060575     180.0      21   9
...              ...    ...           ...            ...       ...  ..  ..  ..
12362  EMF_em_pst_01  1x32p  35683.005250  129540.154750     180.0      01  12
12363  EMF_em_pst_01  1x32p  35689.305250  129540.154750     180.0      01  12
12364  EMF_em_pst_01  1x32p  35701.905250  129540.154750     180.0      01  12
12365  EMF_em_pst_01  1x32p  35708.205250  129540.154750     180.0      01  12
12366  EMF_em_pst_09  1x32p  34127.453648  128845.259495       0.0      09  04

[12367 rows x 8 columns]
                  X       

In [7]:
reinsert_mask = (ELM_df['name'] != '1x96p_ponta') & (ELM_df['name'] != '1x32p') & (ELM_df['rotation'] != 180)
reinsert_df = tracker_df[reinsert_mask].copy()

y_fix = 21.251

reinsert_df['Y'] = np.where(
    reinsert_df['name'] == '1x64p',
    reinsert_df['Y']-y_fix,
    reinsert_df['Y']
)

reinsert_df['new_name'] = np.where(
    reinsert_df['name'] == '1x96p_meio',
    'EMF_tracker_96p',
    'EMF_tracker_64p'
)

reinsert_df['Script'] = reinsert_df.apply(
    lambda row:
    f"._insert {row['new_name']} {row['X']},{row['Y']} 1 1 0", axis=1
)


with open(r"C:\Users\Usuario\Desktop\paiva\Reinserting Trackers.scr", 'w') as f:
    for i in reinsert_df['layer'].unique():
        mask = reinsert_df['layer'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(reinsert_df[mask]['Script'].astype(str)) + '\n')



C:\Users\Usuario\AppData\Local\Temp\ipykernel_18540\1611753215.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reinsert_df = tracker_df[reinsert_mask].copy()
